# Tropical Flower Variety Classification
### 7 classes | ResNet18 vs. EfficientNet-B0 | Frozen vs. fine-tuned

We have ~4,300 photos across 7 tropical flower varieties. Unlike cats vs. dogs vs.
snakes, these classes genuinely look alike to an untrained eye — overlapping colors,
similar petal and leaf shapes. That single fact is why this project needs a bit more
than Project 1: stronger augmentation, and an actual comparison between two backbones
instead of picking one by habit.

**Plan:**
1. Load the data, and split it ourselves (stratified) since it only ships one split.
2. Use stronger augmentation than Project 1, since the classes are harder to tell apart.
3. Run a quick "bake-off" between ResNet18 and EfficientNet-B0 with frozen backbones,
   and let validation accuracy pick the winner.
4. Fine-tune the winner's last block, and measure exactly how much that helps.
5. Evaluate once on the test set, and read the confusion matrix.

**Dataset:** `Project-AgML/tropical_flower_variety_classification` on Hugging Face —
4,319 images, 7 classes. Run this on Google Colab (GPU runtime recommended).

## 1. Setup

In [ ]:
!pip install -q datasets scikit-learn

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


## 2. Load and explore the dataset

This dataset ships only one `"train"` split, so we carve out train/val/test ourselves,
**stratified** so every split keeps roughly the same class balance — with 7 classes a
plain random split could, by chance, under-represent one of them in val or test.

In [ ]:
from datasets import load_dataset

raw = load_dataset("Project-AgML/tropical_flower_variety_classification")
print(raw)

split_1 = raw["train"].train_test_split(test_size=0.30, seed=SEED, stratify_by_column="label")
split_2 = split_1["test"].train_test_split(test_size=0.50, seed=SEED, stratify_by_column="label")
train_ds, val_ds, test_ds = split_1["train"], split_2["train"], split_2["test"]

label_names = raw["train"].features["label"].names
NUM_CLASSES = len(label_names)
print("classes:", label_names)
print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:
counts = np.bincount([train_ds[i]["label"] for i in range(len(train_ds))], minlength=NUM_CLASSES)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(label_names, counts)
ax.set_ylabel("training images"); ax.set_title("Class balance"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()
for name, c in zip(label_names, counts):
    print(f"{name:20s}: {c}")
# If one class has noticeably fewer images, keep that in mind -- it would explain a
# lower recall for that class later, rather than assuming the model is just "bad" at it.

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, idx in zip(axes.flat, random.sample(range(len(train_ds)), 8)):
    ex = train_ds[idx]
    ax.imshow(ex["image"]); ax.set_title(label_names[ex["label"]]); ax.axis("off")
plt.suptitle("Random training samples"); plt.tight_layout(); plt.show()
# Look closely: which varieties share colors or shapes? That is a preview of where the
# confusion matrix will show errors later.


## 3. Preprocessing & a stronger augmentation pipeline

7 visually-similar classes with only ~600 images each is both a harder discrimination
problem and a smaller per-class sample than Project 1, so we go heavier on augmentation:

- **`RandomResizedCrop(224, scale=(0.7, 1.0))`** instead of a plain resize — forces the
  model to recognize a variety from partial views, not just a centered whole flower.
- **Wider `ColorJitter`** — flower photos vary a lot in lighting and white balance across
  cameras; this teaches the model to ignore that instead of treating it as a class cue.
- **`RandomErasing`** — randomly blacks out a small patch, discouraging the model from
  relying on one small, possibly spurious region of the image.

Still ImageNet mean/std (the pretrained backbone expects it), and still **zero**
augmentation on val/test — augmenting them would make the one honest measurement of
"how does the model do on a real photo" noisy and non-reproducible.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.1)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.ds, self.transform = hf_dataset, transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        return self.transform(ex["image"].convert("RGB")), ex["label"]


train_dataset = HFImageDataset(train_ds, train_transform)
val_dataset = HFImageDataset(val_ds, eval_transform)
test_dataset = HFImageDataset(test_ds, eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"train/val/test batches: {len(train_loader)}/{len(val_loader)}/{len(test_loader)}")


## 4. Architecture bake-off: ResNet18 vs. EfficientNet-B0

The brief allows either backbone, so instead of guessing from an ImageNet leaderboard,
we run a quick, cheap frozen-backbone experiment on **our own data** and let validation
accuracy decide — then spend the more expensive fine-tuning compute only on the winner.

| Model | Params | ImageNet top-1 |
|---|---|---|
| ResNet18 | 11.7M | ~69.8% |
| EfficientNet-B0 | 5.3M | ~77.1% (fewer params, higher accuracy via compound scaling) |

EfficientNet-B0 splits a standard convolution into a cheap depthwise step (spatial
mixing) plus a cheap pointwise step (channel mixing), and adds a small
squeeze-and-excitation gate to each block that reweights channels per image — that's
why it gets more accuracy out of fewer parameters than ResNet's plain residual blocks.

In [ ]:
def build_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_efficientnet_b0(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    # EfficientNet's head is model.classifier = Sequential(Dropout, Linear), not a bare
    # `fc` like ResNet -- we rebuild it explicitly so the structure is easy to read.
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.2, inplace=True), nn.Linear(in_features, num_classes))
    return model


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if is_train:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if is_train:
            loss.backward(); optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        n += images.size(0)
    torch.set_grad_enabled(True)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs, lr, params=None, tag=""):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(params if params is not None else model.parameters(), lr=lr)
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader, criterion, None)
        hist["train_loss"].append(tl); hist["train_acc"].append(ta)
        hist["val_loss"].append(vl); hist["val_acc"].append(va)
        print(f"[{tag}] epoch {epoch}/{epochs} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}")
    return hist


In [ ]:
# 4 cheap epochs each, head only, identical setup for both -- a fair, apples-to-apples signal.
resnet = build_resnet18(NUM_CLASSES).to(DEVICE)
hist_resnet_frozen = fit(resnet, train_loader, val_loader, epochs=4, lr=1e-3, tag="ResNet18-frozen")

effnet = build_efficientnet_b0(NUM_CLASSES).to(DEVICE)
hist_effnet_frozen = fit(effnet, train_loader, val_loader, epochs=4, lr=1e-3, tag="EffNet-B0-frozen")

resnet_best_val = max(hist_resnet_frozen["val_acc"])
effnet_best_val = max(hist_effnet_frozen["val_acc"])
print(f"\nResNet18 frozen best val acc:      {resnet_best_val:.4f}")
print(f"EfficientNet-B0 frozen best val acc: {effnet_best_val:.4f}")

winner = "resnet18" if resnet_best_val >= effnet_best_val else "efficientnet_b0"
model = resnet if winner == "resnet18" else effnet
print(f"\n==> Winner (used for fine-tuning below): {winner}")
# We pick the winner by VALIDATION accuracy, not training accuracy or ImageNet rank --
# validation is the only one of these that measures generalization to OUR flower photos.


## 5. Fine-tuning: the frozen-vs-fine-tuned comparison

Unfreeze the winning backbone's last block (`layer4` for ResNet18, or the last two
stages of `features` for EfficientNet-B0) and continue training at a much smaller
learning rate — "nudge, don't overwrite" the pretrained weights, same logic as
Project 1. We keep both the frozen-only and fine-tuned histories so we can report
exactly how many points fine-tuning bought us, not just a final number.

We'd expect this gain to be **larger** here than in Project 1: an easy 3-class task
already had frozen features near their ceiling, but 7 visually-close classes leave
more real headroom for the last block to specialize into.

In [ ]:
def unfreeze_last_block(model, name):
    if name == "resnet18":
        for p in model.layer4.parameters():
            p.requires_grad = True
    else:  # efficientnet_b0
        # EfficientNet-B0's features is 9 narrower stages rather than ResNet's 4 wide
        # blocks, so we unfreeze the last TWO stages to move a comparable amount of the network.
        for stage in list(model.features)[-2:]:
            for p in stage.parameters():
                p.requires_grad = True


unfreeze_last_block(model, winner)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable after unfreeze: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

finetune_params = [p for p in model.parameters() if p.requires_grad]
hist_finetuned = fit(model, train_loader, val_loader, epochs=5, lr=1e-4, params=finetune_params, tag=f"{winner}-finetune")

frozen_hist = hist_resnet_frozen if winner == "resnet18" else hist_effnet_frozen
frozen_best = max(frozen_hist["val_acc"])
finetuned_best = max(hist_finetuned["val_acc"])
print(f"\nFrozen-only best val acc:    {frozen_best:.4f}")
print(f"Fine-tuned best val acc:     {finetuned_best:.4f}")
print(f"Improvement from fine-tuning: {(finetuned_best - frozen_best)*100:+.2f} percentage points")


In [ ]:
labels = ["frozen-only\n(best epoch)", "+fine-tuned\n(best epoch)"]
vals = [frozen_best, finetuned_best]
plt.figure(figsize=(5, 4))
plt.bar(labels, vals, color=["#888", "#4c72b0"])
plt.ylabel("validation accuracy"); plt.ylim(0, 1)
plt.title(f"{winner}: frozen vs. fine-tuned")
for i, v in enumerate(vals):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.show()


## 6. Final evaluation on the held-out test set

We used validation accuracy for every decision so far (which architecture, whether to
fine-tune). The test set gets touched exactly once, right now, so the number we report
is honest — not one that decisions have already been tuned against.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        preds = model(images.to(DEVICE)).argmax(1).cpu()
        all_preds.append(preds); all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


test_preds, test_labels = collect_predictions(model, test_loader)
test_acc = (test_preds == test_labels).mean()
print(f"TEST accuracy ({winner}, fine-tuned): {test_acc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=label_names))

cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(label_names, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(label_names)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix (test set)")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()
# With 7 classes, expect errors to cluster among varieties that actually share color or
# shape, not spread evenly. If two visually SIMILAR varieties are confused, that is a
# reasonable mistake. If two very DIFFERENT-looking varieties are confused often, suspect
# a data problem (mislabeled examples) instead of the model.


## 7. Ideas to improve

- Unfreeze one more block (e.g. `layer3` / stage 6) at an even smaller learning rate —
  worth trying since fine-tuning already showed a real gain here.
- Look at the specific confused pairs from the confusion matrix and target more data or
  augmentation at those varieties specifically, rather than generic across-the-board tweaks.
- Try a higher input resolution (260-300px) if compute allows.
